# NTX Q1 Stage 23 — Formal Monotonic-Authority Property Tests

Empirically checks the paper's central monotonicity property: adding plausible interpretations must never enlarge the common safe core. It also generates explicit counterexamples when interpretation coverage or non-expansive compilation assumptions are violated.

**Evidence rule:** failed/incomplete third-party infrastructure is recorded as such; no missing score is converted into a model failure or a success.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, time, hashlib, zipfile, random
BUNDLE = Path(os.environ.get("NTX_NOTEBOOK_BUNDLE", ".")).resolve()
if str(BUNDLE) not in sys.path:
    sys.path.insert(0, str(BUNDLE))
from ntx_q1_shared import *
ROOT = ensure_dir(DEFAULT_ROOT)
RESULTS_ROOT = ensure_dir(DEFAULT_RESULTS)
SEED = int(os.environ.get("NTX_SEED", "42"))
random.seed(SEED)
print("ROOT:", ROOT)
print("RESULTS:", RESULTS_ROOT)
print("Frozen repo commit:", REPO_COMMIT)

In [ ]:
import numpy as np, pandas as pd
STAGE="23_formal_property_tests"
OUT=ensure_dir(RESULTS_ROOT/STAGE)
rng=np.random.default_rng(SEED)

def safe_core(effect_sets):
    sets=[set(s) for s in effect_sets]
    return set.intersection(*sets) if sets else set()

# Property: I1 subset I2 => K(I2) subset K(I1)
rows=[]
N=int(os.environ.get("NTX_PROPERTY_TRIALS","100000"))
universe=list(range(64))
for trial in range(N):
    k1=int(rng.integers(1,6))
    k2=k1+int(rng.integers(1,5))
    sets=[]
    for _ in range(k2):
        mask=rng.random(len(universe))<rng.uniform(.1,.9)
        sets.append(set(np.array(universe)[mask].tolist()))
    K1=safe_core(sets[:k1])
    K2=safe_core(sets[:k2])
    ok=K2.issubset(K1)
    if not ok or trial<1000:
        rows.append({"trial":trial,"k1":k1,"k2":k2,"K1_n":len(K1),"K2_n":len(K2),"property_holds":ok})
    if not ok:
        break
property_holds=all(r["property_holds"] for r in rows)
pd.DataFrame(rows).to_csv(OUT/"monotonicity_sample_and_failures.csv",index=False)
write_json(OUT/"monotonicity_summary.json",{"trials":N,"property_holds":property_holds})
assert property_holds, "Monotonicity property failed: implementation of test or safe-core construction is wrong."
print("Monotonicity passed",N,"random trials.")

In [ ]:
# Counterexample generator for assumption failures.
# A1 failure: true plausible interpretation omitted.
# A2 failure: compiler grants effect outside the real effect set.
counterexamples=[
    {
        "assumption":"A1 interpretation coverage",
        "represented":[[1,2,3]],
        "omitted_true":[2],
        "authorized_by_represented":[1,2,3],
        "unsafe_effect_example":3,
        "explanation":"An omitted restrictive interpretation can make the represented intersection too large."
    },
    {
        "assumption":"A2 non-expansive effect compilation",
        "real_effects":[1,2],
        "compiled_effects":[1,2,3],
        "unsafe_effect_example":3,
        "explanation":"If the compiler admits an effect not justified by the contract, intersection cannot restore soundness."
    },
    {
        "assumption":"A4 complete mediation",
        "safe_core":[1,2],
        "bypass_effect":9,
        "explanation":"An executor bypass makes the authorization proof irrelevant to the bypassed effect."
    }
]
write_json(OUT/"assumption_counterexamples.json",counterexamples)
record_status(OUT,STAGE,"PASS",random_property_trials=N,monotonicity_verified=True,counterexamples=len(counterexamples))

In [ ]:
z=package_stage(OUT,"NTX_Q1_23_FORMAL_PROPERTY_RESULTS.zip")
print("Created:",z)
maybe_colab_download(z)